# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`

This notebook provides a full walk-through for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following best practices for referencing entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset (this loads and parses the Croissant schema)
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as an object)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available **record sets** and **fields** with their `@id`.

We'll list all record sets and, for each, print its fields (columns) and their IDs.

In [ ]:
print("Available record sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '[no name]')}")

    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields/Columns:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    - @id: {f['@id']} | name: {f.get('name', '[no name]')}")
        else:
            print(f"    - @id: {f}")
    print("")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis. We will use the record set `@id`s and field `@id`s referenced above.

**Tip:** Replace record set IDs below with those shown in the output above for your exploration. We'll attempt to auto-load all available record sets.

In [ ]:
# Build a list of all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"  Loaded {len(df)} records, columns:", df.columns.tolist())
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading record set: {e}")

# If there are successfully loaded frames, preview the first and its columns
if dataframes:
    first_rs = next(iter(dataframes))
    print(f"\nPreview of record set {first_rs} (columns):")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No record sets loaded into dataframes.")

## 4. Exploratory Data Analysis (EDA)

Apply typical data processing: filtering records, normalizing numeric fields, and grouping by important columns. All columns referenced will use their `@id` as per Croissant specification.

First, list numeric fields (if any), and pick a target for the analysis. Edit the below if you wish to use a specific record set and field by `@id`.

In [ ]:
import numpy as np

# Helper: Find numeric columns in all loaded DataFrames
def get_numeric_fields(df):
    numeric_types = [np.dtype('float64'), np.dtype('int64'), np.dtype('float32'), np.dtype('int32')]
    return [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]

selected_record_set_id = None
selected_numeric_field = None

# Select first DataFrame with numeric columns
for rs_id, df in dataframes.items():
    num_fields = get_numeric_fields(df)
    if num_fields:
        selected_record_set_id = rs_id
        selected_numeric_field = num_fields[0]
        print(f"Using record set @id: {rs_id}")
        print(f"Numeric fields: {num_fields}")
        break

if not selected_record_set_id:
    print("No numeric fields found in any record set.")
else:
    threshold = df[selected_numeric_field].quantile(0.90)  # Use 90th percentile as arbitrary example
    print(f"\nFiltering records in {selected_record_set_id} where {selected_numeric_field} > {threshold:.2f}")
    filtered_df = df[df[selected_numeric_field] > threshold]
    print(f"Filtered records with {selected_numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{selected_numeric_field}_normalized"] = (
        (filtered_df[selected_numeric_field] - filtered_df[selected_numeric_field].mean()) /
        filtered_df[selected_numeric_field].std()
    )
    print(f"\nNormalized {selected_numeric_field} for filtered records:")
    display(filtered_df[[selected_numeric_field, f"{selected_numeric_field}_normalized"]].head())

    # Try to group by another column if possible (e.g., the second field if not numeric)
    non_numeric_fields = [col for col in df.columns if col != selected_numeric_field]
    group_field = None
    for col in non_numeric_fields:
        if df[col].dtype == object:
            group_field = col
            break

    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[selected_numeric_field].mean().reset_index()
        print(f"\nGrouped data by {group_field} (mean of {selected_numeric_field}):")
        display(grouped_df.head())
    else:
        print("No suitable non-numeric column found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its normalized values. You may edit below to use your preferred fields.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and selected_numeric_field:
    plt.figure(figsize=(10, 4))
    sns.histplot(filtered_df[selected_numeric_field], kde=True, color='skyblue')
    plt.title(f"Distribution of {selected_numeric_field} (filtered)")
    plt.xlabel(selected_numeric_field)
    plt.show()

    plt.figure(figsize=(10, 4))
    sns.histplot(filtered_df[f"{selected_numeric_field}_normalized"], kde=True, color='coral')
    plt.title(f"Distribution of Normalized {selected_numeric_field} (filtered)")
    plt.xlabel(f"{selected_numeric_field}_normalized")
    plt.show()
else:
    print("Visualization skipped: No numeric data found.")

## 6. Conclusion

In this notebook, we explored the FAIR² dataset—Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya—using the `mlcroissant` library. We loaded metadata, browsed available record sets and fields by `@id`, loaded tabular data, and performed filtering, normalization, and aggregation operations using only Croissant schema IDs. Basic visual analysis revealed the distribution and basic group statistics of the data. Further domain-specific analysis can leverage the rich schema and records loaded through `mlcroissant`.